In [10]:
import numpy as np
import tensorflow as tf
import keras as keras
import os

import pandas as pd
import matplotlib
 
from matplotlib import pyplot
from csv import writer
from tensorflow.keras import layers, models
from keras.callbacks import CSVLogger, EarlyStopping
from data_tools_clipPercent import load_preprocessed, dataPrep, filterReco

In [11]:
# File directory to folder that holds simulation data 
simPrefix = r'C:\Users\mkrit\Documents\RESEARCH\sim_data'

# Sim data to reconstruct (dir produces theta & phi, make sure to transpose)
sim = 'energy'

# Set the number of epochs the model should run for 
numepochs = 50

# Name for model
name = 't_clip9p5__zeroGap'
#t_clip=4

# Baseline data prep
prep = {'q':None, 't':None, 'normed':True, 'reco':'plane', 'cosz':False, 't_shift':True, 't_clip':9.5 }

print(prep)


{'q': None, 't': None, 'normed': True, 'reco': 'plane', 'cosz': False, 't_shift': True, 't_clip': 9.5}


In [12]:
# Add identifying number to name
i = 0
# Saves the h5 file of the model in a folder named models 
while(os.path.exists('models/{}.h5'.format(name+str(i)))): 
    i += 1
name += str(i)
print(name)

t_clip9p5__zeroGap0


In [13]:
# Create model using functional API for multiple inputs

# Input layer 
charge_input = keras.Input(shape=(10,10,4,), name='charge')
#time_input = keras.Input(shape=(10,10,2), name = 'time')

# Starts off with three convolutional layers, each one has half the neurons of the previous one 
conv1_layer = layers.Conv2D(64, kernel_size=3, padding='same', activation='relu')(charge_input)
conv2_layer = layers.Conv2D(32, kernel_size=3, padding='same', activation='relu')(conv1_layer)
conv3_layer = layers.Conv2D(16, kernel_size=3, padding='same', activation="relu")(conv2_layer)

# Layers are flattened before Zenith information is added 
#Flattening is used to convert all the resultant 2-Dimensional arrays from pooled feature maps 
#into a single long continuous linear vector.
flat_layer = layers.Flatten()(conv3_layer)
zenith_input = keras.Input(shape=(1,), name='zenith')
concat_layer = layers.Concatenate()([flat_layer, zenith_input])

# The flattened layers and the Zenith layer run through 3 dense layers
dense1_layer = layers.Dense(256, activation='relu')(concat_layer)
dense2_layer = layers.Dense(256, activation='relu')(dense1_layer)
dense3_layer = layers.Dense(256, activation="relu")(dense2_layer)

# This last dense layer is the output of the model
output = layers.Dense(1)(dense3_layer)

model = models.Model(inputs=[charge_input, zenith_input], outputs=output, name=name)
model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

In [14]:
model.summary()

Model: "t_clip9p5__zeroGap0"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 charge (InputLayer)            [(None, 10, 10, 4)]  0           []                               
                                                                                                  
 conv2d_3 (Conv2D)              (None, 10, 10, 64)   2368        ['charge[0][0]']                 
                                                                                                  
 conv2d_4 (Conv2D)              (None, 10, 10, 32)   18464       ['conv2d_3[0][0]']               
                                                                                                  
 conv2d_5 (Conv2D)              (None, 10, 10, 16)   4624        ['conv2d_4[0][0]']               
                                                                                

In [15]:
# Load simulation data from files for training
x, y = load_preprocessed(simPrefix, 'train')

Percentage of events with a NaN: 2.68


In [16]:
# Prepare event data
x_i = dataPrep(x, y, **prep)

#a = x[0][:,:,[2,3]]


#if (prep['t']!=False):
#for i=0 in 
#t_max = np.amax(x_i[0][:,:,:,2:])
        
#t_min = np.amin((x_i[0][:,:,:,2:]))
#t_data = (x_i[0][:,:,:,2:])
#print(t_data)

#NORMALIZING time:

#x_i[0][:,:,:,2:] = np.clip((x_i[0][:,:,:,2:]),t_min,40000)

#normT = np.log10(1+ np.array(x_i[0][:,:,:,2:]))
#time_data = pd.DataFrame(normT.flatten()) 
#time_data.hist(bins = 100, log = True, edgecolor='none')# range = (.977,1))#tuo = pd.DataFrame(out_two.flatten())
#tuo.hist(bins = 100, log = True, edgecolor='none')


        #print(x[0])
       #print(x[0][:,:,[2,3]])
        #print(x[0][3])
        #print(x[0][:,[2,3]])
#out_max = np.amax(out)

#print(t_min)

#time_data = pd.DataFrame(x_i.flatten())
#time_data.hist(bins= 20,log=True)
# Filter NaNs from reconstruction data



filterReco(prep, y, x_i)

#print(x[0])
#print(x_i)

print(y)
y_data 


In [17]:
# Logs metrics into a csv file between epochs
csv_logger = CSVLogger('models/{}.csv'.format(name))

# Earlystoping stops the model from training when it starts to overfit to the data
# The main parameter we change is the patience 
early_stop = EarlyStopping(monitor="val_loss", min_delta=0, patience=10, verbose=0, mode="auto", baseline=None, restore_best_weights=False) 
callbacks = [early_stop, csv_logger]

# Training
history = model.fit({"charge":x_i[0], "zenith":x_i[1].reshape(-1,1)}, y=y[sim], epochs=numepochs, validation_split=0.15, callbacks=callbacks)

Epoch 1/50
14354/14354 [==============================] - 207s 14ms/step - loss: 0.0763 - mae: 0.1768 - mse: 0.0763 - val_loss: 0.0276 - val_mae: 0.1253 - val_mse: 0.0276
Epoch 2/50
14354/14354 [==============================] - 214s 15ms/step - loss: 0.0310 - mae: 0.1326 - mse: 0.0310 - val_loss: 0.0311 - val_mae: 0.1410 - val_mse: 0.0311
Epoch 3/50
14354/14354 [==============================] - 229s 16ms/step - loss: 0.0277 - mae: 0.1243 - mse: 0.0277 - val_loss: 0.0230 - val_mae: 0.1110 - val_mse: 0.0230
Epoch 4/50
14354/14354 [==============================] - 226s 16ms/step - loss: 0.0261 - mae: 0.1201 - mse: 0.0261 - val_loss: 0.0246 - val_mae: 0.1163 - val_mse: 0.0246
Epoch 5/50
14354/14354 [==============================] - 224s 16ms/step - loss: 0.0252 - mae: 0.1177 - mse: 0.0252 - val_loss: 0.0238 - val_mae: 0.1180 - val_mse: 0.0238
Epoch 6/50
14354/14354 [==============================] - 228s 16ms/step - loss: 0.0246 - mae: 0.1161 - mse: 0.0246 - val_loss: 0.0202 - val_mae:

In [18]:
# Save the model results as a .npy and .h5 file
model.save('Trained/%s.h5' % name)
np.save('Trained/%s.npy' % name, prep)

type(history.history['loss'])
# Get the results of the best epoch and write them to a .csv file
num_epoch=len(history.history['loss'])
val_loss=np.min(history.history['val_loss'])
index=history.history['val_loss'].index(val_loss)
loss=history.history['loss'][index]
new_row=[name, num_epoch, loss, val_loss]
with open('models/results.csv', 'a') as f_object:
    csv_writer=writer(f_object)
    csv_writer.writerow(new_row)
f_object.close() 